**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Physics-Informed Neural Networks

The flagship idea of deep learning *for physics*: instead of fitting data alone, make the network satisfy the **differential equation itself** — autograd computes the derivatives, the PDE residual becomes a loss term, and physics becomes a regularizer that lets you learn from absurdly little data.

## 1. Pre-requisites

- [Intro to PyTorch](./intro_pytorch/intro_pytorch.ipynb) — autograd, training loops.
- An ODE/PDE course helps but the two equations used here are self-contained.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)

---
### 🕐 Session 1 of 2 — *The PDE-as-Loss Idea* (~35 min)
**Goal:** use autograd to differentiate the network w.r.t. its INPUTS; solve an ODE with zero data.
**Builds on:** [Intro to PyTorch](./intro_pytorch/intro_pytorch.ipynb). &nbsp; **Feeds into:** Session 2 (a real boundary-value problem).

---

## 2. Autograd's Second Job

💡 **Intuition.** Backprop differentiates the loss w.r.t. *weights*. But autograd is general: it can just as happily differentiate the network's **output w.r.t. its input** — giving us $u'(t)$, $u''(t)$ for a network $u_\theta(t)$, exactly, no finite differences. So if physics says $u' = -\lambda u$, we can *penalize the network for violating it* at any set of collocation points: $\mathcal{L}_{physics} = \frac{1}{N}\sum_i (u'(t_i) + \lambda u(t_i))^2$. The equation becomes training data — infinite, free, and exact.

In [2]:
# Warm-up: solve u' = -1.5 u, u(0)=1 with NO solution data at all
lam = 1.5
net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))
opt = torch.optim.Adam(net.parameters(), lr=5e-3)

for step in range(2000):
    t = torch.rand(64, 1, requires_grad=True) * 3      # collocation points in [0, 3]
    u = net(t)
    du = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    loss_phys = ((du + lam * u)**2).mean()             # the ODE residual
    u0 = net(torch.zeros(1, 1))
    loss_ic = (u0 - 1.0)**2                            # initial condition
    loss = loss_phys + loss_ic.squeeze()
    opt.zero_grad(); loss.backward(); opt.step()

tt = torch.linspace(0, 3, 200)[:, None]
with torch.no_grad():
    u_hat = net(tt).squeeze()
u_true = np.exp(-lam * tt.squeeze().numpy())
plt.figure(figsize=(7, 2.6))
plt.plot(tt, u_true, "k--", label="analytic  $e^{-1.5t}$")
plt.plot(tt, u_hat, label="PINN (trained on the EQUATION, zero data)")
plt.legend(); plt.tight_layout(); plt.show()
print(f"max error vs analytic solution: {np.abs(u_hat.numpy() - u_true).max():.2e}")

max error vs analytic solution: 5.87e-03


/tmp/ipykernel_2058678/4044410526.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 2 — *A Real Problem: the Damped Oscillator from 6 Points* (~40 min)
**Goal:** combine sparse noisy data with physics; watch the physics term rescue the fit.
**Builds on:** Session 1.

---

## 3. Data + Physics

💡 **Intuition.** The honest selling point of PINNs is the *combination*: a handful of noisy measurements can't pin down a wiggly function — but they can pin down the **constants** (amplitude, phase) of a function the physics already shapes. Loss = data misfit + PDE residual; the physics term acts as an infinitely-informative prior. Watch a plain network hallucinate between 6 points while the PINN interpolates *and extrapolates* correctly.

In [3]:
# damped oscillator: u'' + 2ζω u' + ω² u = 0,  ω=8, ζ=0.05
w0, zeta = 8.0, 0.05
def analytic(t):
    wd = w0 * np.sqrt(1 - zeta**2)
    return np.exp(-zeta*w0*t) * np.cos(wd*t)

t_data = torch.tensor([[0.05], [0.35], [0.61], [0.92], [1.2], [1.53]])   # SIX points
u_data = torch.tensor(analytic(t_data.numpy())) + 0.02*torch.randn(6, 1)

def make_net():
    torch.manual_seed(3)
    return nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
                         nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))

def train_net(physics_weight, steps=4000):
    net = make_net()
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    for s in range(steps):
        loss = ((net(t_data.float()) - u_data.float())**2).mean()
        if physics_weight > 0:
            t = torch.rand(128, 1, requires_grad=True) * 2.0
            u = net(t)
            du = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
            d2u = torch.autograd.grad(du, t, torch.ones_like(du), create_graph=True)[0]
            resid = d2u + 2*zeta*w0*du + w0**2 * u
            loss = loss + physics_weight * (resid**2).mean()
            # initial conditions u(0)=1, u'(0)=0 anchor the family member
            t0 = torch.zeros(1, 1, requires_grad=True)
            u0 = net(t0)
            du0 = torch.autograd.grad(u0, t0, torch.ones_like(u0), create_graph=True)[0]
            loss = loss + (u0 - 1)**2 + du0**2
        opt.zero_grad(); loss.backward(); opt.step()
    return net

net_plain = train_net(0.0)
net_pinn  = train_net(1e-3)

tt = torch.linspace(0, 2, 400)[:, None]
with torch.no_grad():
    up, ui = net_plain(tt).squeeze(), net_pinn(tt).squeeze()
truth = analytic(tt.squeeze().numpy())

plt.figure(figsize=(9, 3))
plt.plot(tt, truth, "k--", linewidth=1, label="true solution")
plt.plot(tt, up, label="plain NN: 6 points, hallucinated physics")
plt.plot(tt, ui, label="PINN: 6 points + the ODE")
plt.plot(t_data, u_data, "ro", markersize=6, label="the six measurements")
plt.legend(fontsize=8); plt.title("physics as a prior: same data, radically different fits")
plt.tight_layout(); plt.show()
print(f"RMSE  plain NN {np.sqrt(np.mean((up.numpy()-truth)**2)):.3f}   PINN {np.sqrt(np.mean((ui.numpy()-truth)**2)):.3f}")

RMSE  plain NN 0.675   PINN 0.027


/tmp/ipykernel_2058678/925058491.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Practicalities worth a slide:** weighting the loss terms is the art (residual and data scales differ — here 1e-3 balances them); stiff/high-frequency problems need Fourier feature inputs or curriculum in time; and PINNs also run *inverse* problems — make $\omega$ a `nn.Parameter` and the same loss estimates the physical constant from data. Try it: you should recover $\omega \approx 8$.

## 4. Conclusion

Autograd differentiates through inputs, so equations become losses, physics becomes a prior, and six noisy points suffice where hundreds were needed. This is the core move of scientific machine learning.

---
## Where next

- [Intro to PyTorch](./intro_pytorch/intro_pytorch.ipynb) — the autograd machinery.
- [Uncertainty in ML](../Intro_Mach_Learn/Uncertainty_in_ML.ipynb) — how much to trust the extrapolation.
- [Kernel Methods](../Intro_Mach_Learn/Kernel_Methods.ipynb) — the classical way of encoding priors, for contrast.